# 04 — Federal Spending vs. Homeless Rate

Does more HUD CoC grant funding per homeless person correlate with lower homeless rates?

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_state_data.csv')
df = df.dropna(subset=['homeless_rate_per_10k', 'spending_per_homeless', 'hud_grants_millions'])
print(f'Loaded {len(df)} states')

Loaded 51 states


In [2]:
slope, intercept, r, p, se = stats.linregress(df['spending_per_homeless'], df['homeless_rate_per_10k'])
r2 = r ** 2
print(f'Spending/homeless vs rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df['spending_per_homeless'].min(), df['spending_per_homeless'].max()]
y_range = [slope * x + intercept for x in x_range]

fig1 = px.scatter(
    df, x='spending_per_homeless', y='homeless_rate_per_10k', text='state_postal',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    title=f'HUD CoC Spending per Homeless Person vs. Homeless Rate by State<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f}</sup>',
    labels={'spending_per_homeless': 'HUD Spending per Homeless Person ($)', 'homeless_rate_per_10k': 'Homeless Rate per 10k'},
    hover_name='state_name',
    hover_data={'hud_grants_millions': ':.1f'},
)
fig1.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
fig1.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig1.show()

Spending/homeless vs rate: r=0.392, R²=0.154, p=0.0044


In [3]:
top15 = df.nlargest(15, 'hud_grants_millions')
fig2 = px.bar(
    top15.sort_values('hud_grants_millions'),
    x='hud_grants_millions', y='state_name', orientation='h',
    color='homeless_rate_per_10k', color_continuous_scale='Reds',
    title='Top 15 States by Total HUD CoC Grant Funding ($ millions)',
    labels={'hud_grants_millions': 'HUD Grants ($M)', 'state_name': ''},
)
fig2.show()